# einops-repeat-broadcast composite — cx5: every-ray-with-every-screen-pixel pairing via repeat + broadcast

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `einops-repeat`, `einops-repeat-broadcast`, `broadcasting-rules`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-repeat-broadcast"
DD_ATOM_IDS = ["einops-repeat", "einops-repeat-broadcast", "broadcasting-rules"]
DD_SUBTOPICS = ["Einops: Repeat", "Einops: Repeat-as-broadcast", "Numpy: Vectorization and broadcasting"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these three atoms compose

ARENA's ray-tracing chapter needs to pair every ray with every screen pixel (or every triangle) without materialising the cross-product. The standard pattern combines three atoms:

1. **`einops-repeat`** — inserts the missing axis via `repeat(x, 'nr d -> nr ns d', ns=NS)`. The output is `(NR, NS, D)`.
2. **`einops-repeat-broadcast`** — the insight that the inserted axis has **stride 0**, so this is a broadcast VIEW, not a copy. `data_ptr()` of the output matches the input.
3. **`broadcasting-rules`** — right-aligned shape arithmetic: `(NR, 1, D)` and `(1, NS, D)` broadcast to `(NR, NS, D)` elementwise.

**The composition.** Two `repeat` calls produce stride-0 broadcast views with matching shapes, then any elementwise op (subtract, multiply, ...) follows numpy broadcasting rules and produces `(NR, NS, D)` outputs.

### Composite Exercise — every-ray-with-every-screen-pixel pairing via repeat + broadcast

**Atoms exercised together**: `einops-repeat`, `einops-repeat-broadcast`, `broadcasting-rules`

Build `cx5_pair_rays_screens(rays, screens)` that takes:
- `rays`: shape `(NR, 3)` — `NR` direction vectors
- `screens`: shape `(NS, 3)` — `NS` screen-pixel positions

and returns the elementwise difference `screens[s] - rays[r]` for every `(r, s)` pair, as a tensor of shape `(NR, NS, 3)`.

Constraints:
- Must use `einops.repeat` to broadcast BOTH inputs to shape `(NR, NS, 3)` first.
- The two intermediate broadcast tensors must be stride-0 views (no copy) — verified via `.data_ptr()`.
- The subtract is allowed to allocate (it produces the output).
- No `.unsqueeze().expand()`, no `torch.broadcast_to`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx5_pair_rays_screens(rays, screens):
    """For every (ray, screen) pair, return screens - rays as (NR, NS, 3)."""
    raise NotImplementedError()

def _test_cx5():
    # --- (a) shape + correctness against a naive loop ---
    NR, NS = 4, 5
    rays = t.randn(NR, 3)
    screens = t.randn(NS, 3)
    out = cx5_pair_rays_screens(rays, screens)
    assert out.shape == (NR, NS, 3), f'expected (NR, NS, 3), got {tuple(out.shape)}'
    for r in range(NR):
        for s in range(NS):
            assert t.allclose(out[r, s], screens[s] - rays[r]), f'mismatch at ({r},{s})'

    # --- (b) hand-built spot-check ---
    rays_h = t.tensor([[1.0, 0.0, 0.0],
                       [0.0, 1.0, 0.0]])      # NR=2
    screens_h = t.tensor([[10.0, 10.0, 10.0],
                          [20.0, 20.0, 20.0],
                          [30.0, 30.0, 30.0]]) # NS=3
    out_h = cx5_pair_rays_screens(rays_h, screens_h)
    assert out_h.shape == (2, 3, 3)
    # ray 0 vs screen 0: (10-1, 10-0, 10-0) = (9, 10, 10)
    assert t.allclose(out_h[0, 0], t.tensor([9.0, 10.0, 10.0]))
    # ray 1 vs screen 2: (30-0, 30-1, 30-0) = (30, 29, 30)
    assert t.allclose(out_h[1, 2], t.tensor([30.0, 29.0, 30.0]))

    # --- (c) scale: no allocation blowup ---
    big_rays = t.randn(2000, 3)
    big_screens = t.randn(100, 3)
    big_out = cx5_pair_rays_screens(big_rays, big_screens)
    assert big_out.shape == (2000, 100, 3)
    # --- atom-coverage: einops.repeat used AND produces stride-0 broadcast views; broadcasting does real work ---
    import inspect as _inspect
    _src = _inspect.getsource(cx5_pair_rays_screens)
    assert 'repeat(' in _src, 'must use einops.repeat (not .expand / broadcast_to)'
    assert '.expand(' not in _src and 'expand_as' not in _src, 'must use einops.repeat, not torch.expand'
    assert 'broadcast_to' not in _src, 'must use einops.repeat, not broadcast_to'
    _g = cx5_pair_rays_screens.__globals__
    _orig_repeat = _g.get('repeat')
    _calls = []
    def _spy_repeat(*a, **kw):
        r = _orig_repeat(*a, **kw)
        _calls.append(r)
        return r
    _g['repeat'] = _spy_repeat
    try:
        cx5_pair_rays_screens(t.randn(3, 3), t.randn(5, 3))
    finally:
        _g['repeat'] = _orig_repeat
    assert len(_calls) >= 1, 'cx5_pair_rays_screens must call einops.repeat at least once'
    assert any(0 in r.stride() for r in _calls), (
        'einops.repeat output must be a stride-0 broadcast view along the inserted pairing axis'
    )
    # broadcasting-rules: shapes differ before repeat (NR vs NS); after repeat both are (NR,NS,3)
    # — verify the result is sensitive to NR != NS (not just shape-matching by accident).
    _out_asym = cx5_pair_rays_screens(t.zeros(2, 3), t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]))
    assert tuple(_out_asym.shape) == (2, 3, 3), f'asymmetric NR,NS must yield (2,3,3); got {tuple(_out_asym.shape)}'
    assert t.allclose(_out_asym[0, 0], t.tensor([1.0, 2.0, 3.0]))
    assert t.allclose(_out_asym[1, 2], t.tensor([7.0, 8.0, 9.0]))

    _dd_passed.add('cx5')

_test_cx5()

<details><summary>Show solution — cx5</summary>

```python
def cx5_pair_rays_screens(rays, screens):
    NR, NS = rays.shape[0], screens.shape[0]
    # repeat inserts a stride-0 axis (the repeat-broadcast atom).
    rays_b    = repeat(rays,    'nr d -> nr ns d', ns=NS)   # (NR, NS, 3)
    screens_b = repeat(screens, 'ns d -> nr ns d', nr=NR)   # (NR, NS, 3)
    # Now broadcasting-rules — both shapes are identical, elementwise sub works.
    return screens_b - rays_b
```

Three atoms in three lines. `repeat(...)` (einops-repeat) is the call; the stride-0 nature of the inserted `ns` / `nr` axis is the repeat-broadcast atom; the final elementwise subtract obeys numpy broadcasting rules — both intermediates have shape `(NR, NS, 3)`, so broadcasting is trivial and the output is `(NR, NS, 3)`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx5',
        'subtopics': ["Einops: Repeat", "Einops: Repeat-as-broadcast", "Numpy: Vectorization and broadcasting"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()